# BronzeToSilver

Notebook de ingestão incremental da camada Silver para a dimensão **tb_channel_group**.

O notebook lê em streaming a tabela Bronze `{CATALOG}.bronze.tb_channel_group` e aplica um **MERGE (upsert)** na tabela Silver `{CATALOG}.silver.tb_channel_group`, usando `trade_chnl_desc` como chave.

## Parâmetros

| Parâmetro | Descrição |
|---|---|
| `CATALOG` | Catálogo do Unity Catalog de origem/destino |
| `TABLE_NAME` | Nome da tabela |
| `CHECKPOINT_PATH` | Caminho do checkpoint do stream |

## Fluxo

1. **Setup** — define widgets e monta os nomes de origem/target.
2. **Reset** — apaga a tabela Silver existente e o checkpoint (reprocessamento completo).
3. **Transform** — remove espaços dos campos descritivos e mantém apenas a linha mais recente por canal (`row_number` por `_ingestion_timestamp` / `_source_file`).
4. **Ingestão** — `readStream` com `foreachBatch`: na primeira execução cria a tabela; nas seguintes faz `MERGE` (`whenMatchedUpdateAll` / `whenNotMatchedInsertAll`) com `trigger(availableNow=True)`.

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("CATALOG", "")
dbutils.widgets.text("CHECKPOINT_PATH", "")

CATALOG = dbutils.widgets.get("CATALOG")
CHECKPOINT_PATH = dbutils.widgets.get("CHECKPOINT_PATH")

SOURCE = f"{CATALOG}.bronze.tb_channel_group"
TARGET = f"{CATALOG}.silver.tb_channel_group"  
KEY = "trade_chnl_desc"


In [0]:

spark.sql(f"DROP TABLE IF EXISTS {TARGET}")
dbutils.fs.rm(CHECKPOINT_PATH, True)

In [0]:
def transform(df):
    """Remove espaços e mantém 1 linha por canal (a mais recente)."""
    out = df.select(
        F.trim("trade_chnl_desc").alias("trade_chnl_desc"),
        F.trim("trade_group_desc").alias("trade_group_desc"),
        F.trim("trade_type_desc").alias("trade_type_desc"),
        "_source_file",
        "_ingestion_timestamp",
    )
    w = Window.partitionBy(KEY).orderBy(F.col("_ingestion_timestamp").desc(), F.col("_source_file").desc())
    return out.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
def upsert_batch(batch_df, batch_id):
    from delta.tables import DeltaTable
    df = transform(batch_df)
    if not spark.catalog.tableExists(TARGET):
        df.write.format("delta").saveAsTable(TARGET)
        return
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(df.alias("s"), f"t.{KEY} <=> s.{KEY}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


(spark.readStream.table(SOURCE)
      .writeStream.foreachBatch(upsert_batch)
      .option("checkpointLocation", CHECKPOINT_PATH)
      .trigger(availableNow=True)
      .start()
      .awaitTermination())